# seq2seq(use rnn)

同用例对照：
- [seq2seq(use lstm)](./25_lstm_seq2seq.ipynb)
- [seq2seq(use Transformer)](./06_pytorch_transformer.ipynb)

---

## 1. 一句话目标

中文句子 → Encoder（**nn.RNN**）压成隐状态 → Decoder（**nn.RNN**）按已生成英文逐步预测下一个词，直到结束符 `E`。

与 Transformer 版同一任务与同一三句语料；本篇**不加 Attention**，只看循环骨干。

---

## 2. 数据流

```
sentences → make_data（P/S/E 与 06 相同）→ enc_inputs / dec_inputs / dec_outputs
  → DataLoader
  → Encoder: Embedding → nn.RNN → 最终隐状态
  → Decoder: Embedding → nn.RNN(以 Encoder 隐状态为初态) → Linear → 词表 logits
  → CrossEntropyLoss(ignore_index=0)
```

---

## 3. 模块结构

```
Seq2Seq
├── Encoder
│   ├── src_emb (Embedding)
│   └── rnn (nn.RNN)
└── Decoder
    ├── tgt_emb (Embedding)
    ├── rnn (nn.RNN)
    └── projection (Linear → tgt_vocab_size)
```

---

## 4. 张量形状要点

| 变量 | 形状 | 说明 |
|------|------|------|
| enc_inputs / dec_inputs | (B, L) | 词 id |
| Embedding 后 | (B, L, emb) | |
| RNN 输出 | (B, L, hidden) | `batch_first=True` |
| 最终隐状态 | (1, B, hidden) | 传给 Decoder 作初态 |
| logits | (B, L, V) → view(-1, V) | 与 dec_outputs 算 CE |


## 5. 依赖

PyTorch（`torch`）。CPU 即可。


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as Data


## 6. 语料与词表（与 06 相同）


In [ ]:
# 与 06_pytorch_transformer / 另外两篇 seq2seq 完全同一用例
sentences = [
    ['我 是 教 师 P', 'S I am a teacher', 'I am a teacher E'],
    ['我 喜 欢 教 学', 'S I like teaching P', 'I like teaching P E'],
    ['我 是 厨 师 P', 'S I am a cook', 'I am a cook E'],
]

src_vocab = {'P': 0, '我': 1, '是': 2, '教': 3, '师': 4, '喜': 5, '欢': 6, '学': 7, '厨': 8}
src_idx2word = {i: w for w, i in src_vocab.items()}
src_vocab_size = len(src_vocab)

tgt_vocab = {'P': 0, 'S': 1, 'E': 2, 'I': 3, 'am': 4, 'a': 5, 'teacher': 6, 'like': 7, 'teaching': 8, 'cook': 9}
idx2word = {i: w for w, i in tgt_vocab.items()}
tgt_vocab_size = len(tgt_vocab)

src_len = len(sentences[0][0].split())
tgt_len = len(sentences[0][1].split())
print('src_len', src_len, 'tgt_len', tgt_len)


In [ ]:
def make_data(sentences):
    enc_inputs, dec_inputs, dec_outputs = [], [], []
    for i in range(len(sentences)):
        enc_input = [[src_vocab[n] for n in sentences[i][0].split()]]
        dec_input = [[tgt_vocab[n] for n in sentences[i][1].split()]]
        dec_output = [[tgt_vocab[n] for n in sentences[i][2].split()]]
        enc_inputs.extend(enc_input)
        dec_inputs.extend(dec_input)
        dec_outputs.extend(dec_output)
    return (
        torch.LongTensor(enc_inputs),
        torch.LongTensor(dec_inputs),
        torch.LongTensor(dec_outputs),
    )

enc_inputs, dec_inputs, dec_outputs = make_data(sentences)
print('enc_inputs:')
print(enc_inputs)
print('dec_inputs:')
print(dec_inputs)
print('dec_outputs:')
print(dec_outputs)


In [ ]:
class MyDataSet(Data.Dataset):
    def __init__(self, enc_inputs, dec_inputs, dec_outputs):
        self.enc_inputs = enc_inputs
        self.dec_inputs = dec_inputs
        self.dec_outputs = dec_outputs

    def __len__(self):
        return self.enc_inputs.shape[0]

    def __getitem__(self, idx):
        return self.enc_inputs[idx], self.dec_inputs[idx], self.dec_outputs[idx]

loader = Data.DataLoader(MyDataSet(enc_inputs, dec_inputs, dec_outputs), batch_size=2, shuffle=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)


## 7. 模型：Encoder / Decoder（nn.RNN）


In [ ]:
emb_size = 64
hidden_size = 128

class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.src_emb = nn.Embedding(src_vocab_size, emb_size)
        self.rnn = nn.RNN(emb_size, hidden_size, batch_first=True)

    def forward(self, enc_inputs):
        # enc_inputs: (B, src_len)
        embedded = self.src_emb(enc_inputs)  # (B, src_len, emb)
        outputs, h = self.rnn(embedded)      # h: (1, B, hidden)
        return h


class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.tgt_emb = nn.Embedding(tgt_vocab_size, emb_size)
        self.rnn = nn.RNN(emb_size, hidden_size, batch_first=True)
        self.projection = nn.Linear(hidden_size, tgt_vocab_size)

    def forward(self, dec_inputs, h):
        # dec_inputs: (B, tgt_len); teacher forcing
        embedded = self.tgt_emb(dec_inputs)
        outputs, h = self.rnn(embedded, h)
        logits = self.projection(outputs)  # (B, tgt_len, V)
        return logits, h


class Seq2Seq(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = Encoder()
        self.decoder = Decoder()

    def forward(self, enc_inputs, dec_inputs):
        h = self.encoder(enc_inputs)
        logits, _ = self.decoder(dec_inputs, h)
        return logits


## 8. 训练（teacher forcing）


In [ ]:
torch.manual_seed(0)
model = Seq2Seq().to(device)
criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = optim.Adam(model.parameters(), lr=1e-2)

for epoch in range(400):
    total = 0.0
    for enc, dec_in, dec_out in loader:
        enc, dec_in, dec_out = enc.to(device), dec_in.to(device), dec_out.to(device)
        logits = model(enc, dec_in)  # (B, L, V)
        loss = criterion(logits.reshape(-1, tgt_vocab_size), dec_out.reshape(-1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item()
    if (epoch + 1) % 40 == 0:
        print(f'epoch {epoch+1:3d}  loss={total/len(loader):.4f}')


## 9. 贪婪解码试译


In [ ]:
@torch.no_grad()
def greedy_decode(model, enc_sentence, max_len=tgt_len):
    """enc_sentence: 空格分词的中文，如 '我 是 教 师 P'"""
    model.eval()
    ids = [src_vocab[w] for w in enc_sentence.split()]
    enc = torch.LongTensor([ids]).to(device)
    h = model.encoder(enc)

    dec_token = torch.LongTensor([[tgt_vocab['S']]]).to(device)
    result = []
    for _ in range(max_len):
        logits, h = model.decoder(dec_token, h)
        next_id = int(logits[0, -1].argmax())
        if next_id == tgt_vocab['E']:
            break
        if next_id != tgt_vocab['P']:
            result.append(idx2word[next_id])
        dec_token = torch.LongTensor([[next_id]]).to(device)
    return ' '.join(result)

for s in ['我 是 教 师 P', '我 喜 欢 教 学', '我 是 厨 师 P']:
    print(s, '→', greedy_decode(model, s))


## 10. 已知限制与后续

- 无 Attention：长句信息全挤在最终隐状态里。
- 小语料过拟合训练集即可，不追求泛化。
- 对照：同用例下换 LSTM / Transformer 骨干，看结构与收敛差异。
